# Multi-Agent Research System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genieincodebottle/aiml-companion/blob/main/projects/ai-agents-project/notebooks/Multi_Agent_Research.ipynb)

> LangGraph pipeline: Researcher -> Analyst -> Writer -> Fact-Checker with guardrails.

**Requires:**
- **OpenAI API key** - [Get one here](https://platform.openai.com/api-keys) (requires billing, ~$0.01 per pipeline run with gpt-4o-mini)
- **Tavily API key** - [Get one here](https://app.tavily.com/) (free tier: 1,000 searches/month)

**Run cells in order from top to bottom.** Each cell depends on the previous ones.

## Setup


In [ ]:
!pip install langgraph langchain-openai langchain-community tavily-python requests -q

## Step 1: Set API Keys

In [ ]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
os.environ["TAVILY_API_KEY"] = getpass("Tavily API key: ")

# Validate keys before proceeding
assert os.environ["OPENAI_API_KEY"].startswith("sk-"), "Invalid OpenAI key - should start with sk-"
assert len(os.environ["TAVILY_API_KEY"]) > 10, "Tavily key looks too short"
print("Keys set and validated!")


## Step 2: Guardrails

In [ ]:
import re, requests, logging
logger = logging.getLogger(__name__)

PII_PATTERNS = {
    "email": re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"),
    "phone": re.compile(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b"),
    "ssn": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
}
TOKEN_BUDGET = 50000

def detect_pii(text):
    return {t: p.findall(text) for t, p in PII_PATTERNS.items() if p.findall(text)}

def scrub_pii(text):
    found = []
    for t, p in PII_PATTERNS.items():
        if p.search(text):
            found.append(t)
            text = p.sub(f"[REDACTED_{t.upper()}]", text)
    return text, found

def check_budget(tokens, budget=TOKEN_BUDGET):
    return tokens < budget

def validate_url(url, timeout=5):
    try:
        return requests.head(url, timeout=timeout, allow_redirects=True).status_code < 400
    except:
        return False

# Test
print(f"PII: {detect_pii('john@test.com 555-123-4567')}")
print(f"Budget (1K): {check_budget(1000)}, Budget (60K): {check_budget(60000)}")

## Step 3: Agent Nodes + Pipeline

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

class ResearchState(TypedDict):
    query: str
    sources: list[dict]
    key_claims: list[dict]
    draft: str
    fact_check: list[dict]
    final_report: str
    token_count: int
    errors: list[str]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
search = TavilySearchResults(max_results=5)

def researcher(state):
    if not check_budget(state.get("token_count", 0)):
        return {"errors": state.get("errors", []) + ["Budget exceeded"]}
    results = search.invoke(state["query"])
    sources = [{"title": r.get("title",""), "url": r["url"], "snippet": r["content"][:500]} for r in results]
    return {"sources": sources, "token_count": state.get("token_count", 0) + 500}

def analyst(state):
    if not state.get("sources"):
        print("WARNING: No sources found. The writer will produce an empty report.")
        return {"key_claims": [], "errors": state.get("errors", []) + ["No sources returned by researcher"]}
    text = "\n".join(f"[{i+1}] {s['title']}: {s['snippet']}" for i, s in enumerate(state["sources"]))
    resp = llm.invoke(f"Extract 5-8 key claims. Format: claim | source | confidence\n\n{text}")
    claims = [{"claim": p[0].strip(), "source_idx": p[1].strip(), "confidence": p[2].strip() if len(p)>2 else "medium"}
              for line in resp.content.split("\n") for p in [line.split("|")] if len(p) >= 2]
    tokens = resp.response_metadata.get("token_usage", {}).get("total_tokens", 1000)
    return {"key_claims": claims, "token_count": state.get("token_count", 0) + tokens}

def writer(state):
    claims = "\n".join(f"- {c['claim']} [Source {c['source_idx']}]" for c in state["key_claims"])
    sources = "\n".join(f"[{i+1}] {s['title']} - {s['url']}" for i, s in enumerate(state["sources"]))
    resp = llm.invoke(f"Write a research report using ONLY these claims.\n\nClaims:\n{claims}\n\nSources:\n{sources}")
    tokens = resp.response_metadata.get("token_usage", {}).get("total_tokens", 1500)
    return {"draft": resp.content, "token_count": state.get("token_count", 0) + tokens}

def fact_checker(state):
    summaries = "\n".join(f"[{i+1}] {s['snippet'][:200]}" for i, s in enumerate(state["sources"]))
    check = llm.invoke(f"Verify claims. For unsupported: UNSUPPORTED: <claim>\n\nReport:\n{state['draft'][:2000]}\n\nSources:\n{summaries}")
    revised = state["draft"]
    for line in check.content.split("\n"):
        if line.strip().startswith("UNSUPPORTED:"):
            claim = line.replace("UNSUPPORTED:", "").strip()
            if claim and claim in revised:
                revised = revised.replace(claim, "[NEEDS CITATION] " + claim)
    revised, pii = scrub_pii(revised)
    tokens = check.response_metadata.get("token_usage", {}).get("total_tokens", 1000)
    return {"final_report": revised, "token_count": state.get("token_count", 0) + tokens}

# Build graph
graph = StateGraph(ResearchState)
for name, fn in [("researcher", researcher), ("analyst", analyst), ("writer", writer), ("fact_checker", fact_checker)]:
    graph.add_node(name, fn)
graph.set_entry_point("researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "writer")
graph.add_edge("writer", "fact_checker")
graph.add_edge("fact_checker", END)
app = graph.compile()
print("Pipeline built: researcher -> analyst -> writer -> fact_checker")

## Step 4: Run the Pipeline

In [ ]:
result = app.invoke({
    "query": "What are the latest trends in AI agents for 2025?",
    "sources": [], "key_claims": [], "draft": "",
    "fact_check": [], "final_report": "",
    "token_count": 0, "errors": []
})

print(f"Sources: {len(result['sources'])}")
print(f"Claims: {len(result['key_claims'])}")
print(f"Tokens: {result['token_count']} / {TOKEN_BUDGET}")
print(f"\n{'='*60}\nFINAL REPORT:\n{'='*60}")
print(result['final_report'][:2000])

## Key Takeaways

- **LangGraph StateGraph**: Shared TypedDict state flows between agents
- **Guardrails**: PII scrubbing + budget enforcement at every step
- **Fact-checker**: Flags unsupported claims with [NEEDS CITATION]
- **Error handling**: try/except in every node prevents pipeline crashes
- **Token budget**: 50K hard limit prevents runaway costs